# Public Pipeline: Wellbore Geology Prediction

Этот блокнот будем строить по этапам. Каждый этап — отдельный смысловой блок: сначала делаем надежную основу проекта, потом загрузчик данных, baseline, физические кандидаты, ML-модели и stacking.


## Kaggle-config & checking dataset

In [ ]:

from dataclasses import dataclass
from pathlib import Path

import pandas as pd


'''
Stage 1 builds the Kaggle-first project configuration and a lightweight dataset inventory.
It does not train models or read the full dataset into memory.
'''


@dataclass(frozen=True)
class ProjectConfig:
    project_dir: Path
    dataset_dir: Path
    train_dir: Path
    test_dir: Path
    sample_submission_path: Path
    work_dir: Path
    seed: int = 42
    n_splits: int = 5

    @classmethod
    def for_kaggle(cls) -> "ProjectConfig":
        kaggle_dataset_dir = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
        current_dir = Path.cwd().resolve()
        local_project_dir = current_dir if (current_dir / "datasets").exists() else current_dir.parent
        local_dataset_dir = local_project_dir / "datasets"

        if kaggle_dataset_dir.exists():
            project_dir = Path("/kaggle/working")
            dataset_dir = kaggle_dataset_dir
            work_dir = Path("/kaggle/working")
        else:
            project_dir = local_project_dir
            dataset_dir = local_dataset_dir
            work_dir = project_dir / "working"

        return cls(
            project_dir=project_dir,
            dataset_dir=dataset_dir,
            train_dir=dataset_dir / "train",
            test_dir=dataset_dir / "test",
            sample_submission_path=dataset_dir / "sample_submission.csv",
            work_dir=work_dir,
        )

    def validate(self) -> None:
        required_paths = [
            self.dataset_dir,
            self.train_dir,
            self.test_dir,
            self.sample_submission_path,
        ]
        missing_paths = [path for path in required_paths if not path.exists()]
        if missing_paths:
            readable = "\n".join(str(path) for path in missing_paths)
            raise FileNotFoundError(f"Missing required dataset paths:\n{readable}")
        self.work_dir.mkdir(parents=True, exist_ok=True)


class DatasetInventory:
    def __init__(self, config: ProjectConfig):
        self.config = config

    def horizontal_files(self, split: str) -> list[Path]:
        return sorted((self.config.dataset_dir / split).glob("*__horizontal_well.csv"))

    def typewell_files(self, split: str) -> list[Path]:
        return sorted((self.config.dataset_dir / split).glob("*__typewell.csv"))

    def png_files(self, split: str) -> list[Path]:
        return sorted((self.config.dataset_dir / split).glob("*.png"))

    def well_ids(self, split: str) -> set[str]:
        return {path.name.split("__")[0] for path in self.horizontal_files(split)}

    def overview(self) -> pd.DataFrame:
        rows = []
        for split in ["train", "test"]:
            horizontal_wells = self.well_ids(split)
            typewell_wells = {path.name.split("__")[0] for path in self.typewell_files(split)}
            rows.append({
                "split": split,
                "horizontal_files": len(self.horizontal_files(split)),
                "typewell_files": len(self.typewell_files(split)),
                "png_files": len(self.png_files(split)),
                "wells_with_both_csvs": len(horizontal_wells & typewell_wells),
                "horizontal_only_wells": len(horizontal_wells - typewell_wells),
                "typewell_only_wells": len(typewell_wells - horizontal_wells),
            })
        return pd.DataFrame(rows)

    def schema_examples(self) -> pd.DataFrame:
        rows = []
        for split in ["train", "test"]:
            examples = {
                "horizontal": self.horizontal_files(split),
                "typewell": self.typewell_files(split),
            }
            for file_type, files in examples.items():
                if not files:
                    continue
                frame = pd.read_csv(files[0], nrows=5)
                rows.append({
                    "split": split,
                    "file_type": file_type,
                    "example_file": files[0].name,
                    "columns": ", ".join(frame.columns),
                    "preview_rows": len(frame),
                })
        return pd.DataFrame(rows)

    def submission_overview(self) -> pd.DataFrame:
        sample_submission = pd.read_csv(self.config.sample_submission_path)
        well_ids = sample_submission["id"].str.split("_", n=1).str[0]
        return pd.DataFrame([{
            "rows": len(sample_submission),
            "columns": ", ".join(sample_submission.columns),
            "wells": well_ids.nunique(),
            "first_id": sample_submission["id"].iloc[0],
            "last_id": sample_submission["id"].iloc[-1],
        }])


config = ProjectConfig.for_kaggle()
config.validate()
inventory = DatasetInventory(config)

print(f"Project directory: {config.project_dir}")
print(f"Dataset directory: {config.dataset_dir}")
print(f"Working directory: {config.work_dir}")

display(inventory.overview())
display(inventory.schema_examples())
display(inventory.submission_overview())
